# Audio processing and voiceprint verification

**Owner:** Towi (Audio lead) · branch `feat/audio`

The voice model is the system's final gate. After the face model recognises someone, their voice has to (1) be confidently recognised and (2) belong to **the same member** before the product recommendation is shown.

| Section | What it covers | Rubric criterion |
|---|---|---|
| 1 | Inventory of every recording | Audio Quality |
| 2 | Loading and preprocessing | Audio Quality |
| 3 | Waveform + spectrogram per member | Audio Quality & Visualization |
| 4 | Augmentation | Audio Augmentation |
| 5 | Feature extraction to `audio_features.csv` | Audio Feature Extraction |
| 6 | Voice model and evaluation | Model Implementation, Evaluation |
| 7 | Confidence threshold with outsiders | Multimodal Logic |
| 8 | Save the model, test it as the app will use it | System Simulation |

The reusable code lives in `src/audio_pipeline.py`, so the command-line app uses exactly the same steps as this notebook.

In [ ]:
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import soundfile as sf
import librosa, librosa.display
from sklearn.metrics import ConfusionMatrixDisplay

from src import audio_pipeline as ap

FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")
print("Audio folder:", ap.AUDIO_DIR)

## 1. Inventory: what did each phone record?

A recording is a long list of numbers: the air pressure at each instant. The **sample rate** is how many of those snapshots are taken per second. Think of a flipbook: 16 kHz means 16,000 pages a second, 48 kHz means 48,000. Different phones and apps (voice recorder, WhatsApp) use different rates and channel counts. Before comparing four people's voices, we check that we know exactly what we've got.

`soundfile.info` reads a file's header without loading the sound. A file that can't be read here is usually a compressed voice note that was renamed to `.wav` without being converted.

In [ ]:
rows = []
for path in sorted(ap.AUDIO_DIR.glob("*/*.wav")):
    try:
        info = sf.info(path)
        rows.append({"member": path.parent.name, "file": path.name, "sample_rate": info.samplerate,
                     "channels": info.channels, "duration_s": round(info.duration, 2), "readable": True})
    except Exception as err:
        rows.append({"member": path.parent.name, "file": path.name, "readable": False, "error": str(err)[:60]})

inventory = pd.DataFrame(rows)
display(inventory)
print(inventory.groupby("member").size().rename("clips per member").to_string())

**Interpretation:** _(fill in after running)_ How many clips does each member have? Do the sample rates and channel counts differ between members? If they do, that's the reason for section 2: resample everything to one standard format.

## 2. Loading and preprocessing

`ap.load_clip()` does three things to every clip, in this order:

1. **Resample to 16 kHz mono.** Every flipbook is redrawn at the same speed, as one channel. 16 kHz covers the full range of human speech (phone calls use 8 kHz), so nothing useful is lost, and the files stay small.
2. **Trim silence** (`librosa.effects.trim`, `top_db=20`). Anything more than 20 dB quieter than the loudest point at the start and end is cut. We tested 30, 25 and 20 dB on real clips: phone background noise sits only about 30 dB below speech, so 30 dB trimmed almost nothing. At 20 dB every clip was cut cleanly to where the speech starts (about 0.7 to 1 s in). The trade-off is that the very softest end of a word can be clipped, which carries little information about who is speaking. Without this, the second of silence before someone speaks would be averaged into their "voice" features, and people who paused longer would look different for the wrong reason.
3. **Peak-normalise the volume** (loudest point = 0.9). How loud a clip is depends mostly on how close the phone was, not on who's speaking. If we left it, the model could learn "loud = Kethia", which breaks the moment she holds her phone further away. The 0.9 (not 1.0) leaves headroom so adding noise in section 4 doesn't distort the sound.

The plot below shows one clip before and after preprocessing.

In [ ]:
example = sorted(ap.AUDIO_DIR.glob("*/*yes_approve.wav"))[0]
raw, sr_raw = librosa.load(example, sr=None, mono=True)
clean = ap.load_clip(example)

fig, axes = plt.subplots(1, 2, figsize=(13, 3), sharey=True)
librosa.display.waveshow(raw, sr=sr_raw, ax=axes[0])
axes[0].set(title=f"Raw: {example.name} ({sr_raw} Hz, {len(raw)/sr_raw:.2f} s)", xlabel="Time (s)", ylabel="Amplitude")
librosa.display.waveshow(clean, sr=ap.SR, ax=axes[1], color="tab:orange")
axes[1].set(title=f"Preprocessed: 16 kHz, trimmed, normalised ({len(clean)/ap.SR:.2f} s)", xlabel="Time (s)")
plt.tight_layout(); plt.savefig(FIG_DIR / "audio_preprocessing.png", dpi=150); plt.show()

**Interpretation:** _(fill in after running)_ How much silence was trimmed? How did the height of the wave change after normalising?

## 3. Waveform and spectrogram for each member

The two views show different things:

- **Waveform:** loudness over time. You can see where each word starts and stops, but not *what the sound is made of*.
- **Mel spectrogram:** which pitches are present at each moment, like a piano roll. Time runs left to right, pitch goes up, and brightness is how strong that pitch is. The mel scale spaces the pitches the way human ears hear them: lots of detail in the low range where voices live, less in the high range.

Horizontal stripes in voiced sounds are **harmonics**: the base pitch of the voice and its multiples. Their spacing is set by how fast the vocal cords vibrate, which is one of the things that makes each voice different.

In [ ]:
members = sorted(p.name for p in ap.AUDIO_DIR.iterdir() if p.is_dir())
fig, axes = plt.subplots(len(members), 2, figsize=(14, 3 * len(members)))
for row, member in zip(np.atleast_2d(axes), members):
    path = sorted((ap.AUDIO_DIR / member).glob("*yes_approve.wav"))[0]
    y = ap.load_clip(path)
    librosa.display.waveshow(y, sr=ap.SR, ax=row[0])
    row[0].set(title=f"{member}: waveform ('Yes, approve')", xlabel="Time (s)", ylabel="Amplitude")
    mel_db = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=ap.SR, n_mels=64), ref=np.max)
    img = librosa.display.specshow(mel_db, sr=ap.SR, x_axis="time", y_axis="mel", ax=row[1])
    row[1].set(title=f"{member}: mel spectrogram", xlabel="Time (s)", ylabel="Frequency (Hz, mel scale)")
    fig.colorbar(img, ax=row[1], format="%+2.0f dB")
plt.tight_layout(); plt.savefig(FIG_DIR / "audio_waveforms_spectrograms.png", dpi=150); plt.show()

**Interpretation:** _(fill in after running, 3 to 4 sentences)_
- Can you see the two words "yes" and "approve" as separate bursts in each waveform?
- In the spectrograms, where is most of the energy (low or high frequencies)? Whose harmonic stripes are spaced further apart (a higher voice)?
- Which differences between members look like *voice* differences, and which look like *recording* differences (background noise as a haze across the whole spectrogram, for example)?

## 4. Augmentation

With 4 recordings per person, the model sees very little variety. Augmentation creates realistic variations of each recording, so the model learns what stays constant about a voice instead of memorising four specific clips.

| Augmentation | What it simulates | Why it's kept small |
|---|---|---|
| Pitch +1 / -1 semitone | a slightly higher or lower voice (tired, excited) | A large shift makes you sound like a different person, and pitch is part of what identifies you. One semitone is about the natural day-to-day variation. |
| Speed x1.1 / x0.9 | talking a bit faster or slower, with pitch unchanged | Speed changes the timing but not the voice itself. |
| Background noise at 20 dB SNR | a noisier room | The speech stays 100 times more powerful than the noise, which sounds like a busy room, not a broken recording. |

So each recording becomes 6 versions (original + 5). That's **5 augmentations per sample**; the rubric asks for at least 2.

One important rule follows from this: an augmented copy is almost the same recording. If the original is in training and its copy in testing, the test is too easy. Section 6 keeps every recording and all its copies on the same side of the split.

In [ ]:
y = ap.load_clip(example)
versions = ap.augment(y, seed=0)
fig, axes = plt.subplots(2, 3, figsize=(15, 6.5), sharey=True)
for ax, (name, y_aug) in zip(axes.flat, versions.items()):
    mel_db = librosa.power_to_db(librosa.feature.melspectrogram(y=y_aug, sr=ap.SR, n_mels=64), ref=np.max)
    librosa.display.specshow(mel_db, sr=ap.SR, x_axis="time", y_axis="mel", ax=ax)
    ax.set(title=f"{name} ({len(y_aug)/ap.SR:.2f} s)", xlabel="Time (s)", ylabel="Hz (mel)")
plt.suptitle(f"Augmentations of {example.name}", y=1.0)
plt.tight_layout(); plt.savefig(FIG_DIR / "audio_augmentations.png", dpi=150); plt.show()

**Interpretation:** _(fill in after running)_ Compare each version with the original. Pitch shifts move the harmonic stripes slightly up or down. Speed changes stretch or squeeze the clip in time. Noise adds an even, speckled texture across all frequencies and drowns out the quietest details of the speech. The overall pattern of the words should still be recognisable in every version, which is the point.

## 5. Feature extraction

Models like logistic regression need every example to be the same list of numbers, but clips have different lengths. So we compute features on short frames (32 ms each) and then summarise each feature over the whole clip with its **mean** (typical value) and **standard deviation** (how much it moves around). It's like describing a song by its overall mood and how much it varies, instead of listing every note.

| Feature | What it measures | Columns |
|---|---|---|
| **MFCC 1 to 13** | The shape of the vocal tract. Your throat and mouth act like an instrument and the words are the song. MFCCs describe the instrument, which is why they identify a speaker whatever phrase they say. | mean + std (26) |
| **MFCC delta** | How quickly that shape changes, which reflects speaking style | std (13) |
| **Spectral roll-off** | The frequency below which 85% of the energy sits: how "bright" or "dark" a voice is | mean + std |
| **Spectral centroid** | The "centre of mass" of the frequencies | mean + std |
| **RMS energy** | Loudness per frame. Its *variation* (std) shows how much someone stresses words, even after normalising. | mean + std |
| **Zero-crossing rate** | How often the wave crosses zero, which is high for hissy sounds like "s" and "f" | mean + std |

That's 47 features per row, plus 4 label columns: `member`, `phrase`, `augmentation`, and `source_audio` (the original recording each row came from).

In [ ]:
features = ap.build_feature_table()
ap.FEATURES_CSV.parent.mkdir(parents=True, exist_ok=True)
features.to_csv(ap.FEATURES_CSV, index=False)
print(f"Saved {features.shape[0]} rows x {features.shape[1]} columns -> {ap.FEATURES_CSV.relative_to(ROOT)}")
print("Rows per member:", features["member"].value_counts().to_dict())
display(features.head())

In [ ]:
mfcc_means = (features[features["augmentation"] == "original"]
              .groupby("member")[[f"mfcc_{i}_mean" for i in range(1, 14)]].mean())
fig, ax = plt.subplots(figsize=(10, 4))
for member, values in mfcc_means.iterrows():
    ax.plot(range(1, 14), values.values, marker="o", label=member)
ax.set(title="Average MFCC profile per member (original recordings)", xlabel="MFCC coefficient", ylabel="Mean value")
ax.legend(); plt.tight_layout(); plt.savefig(FIG_DIR / "audio_mfcc_profiles.png", dpi=150); plt.show()

**Interpretation:** _(fill in after running)_ MFCC 1 mostly reflects overall energy. The coefficients after it describe the shape of the voice. Where do the members' lines separate most clearly? Where they overlap, the model will find it harder to tell those people apart.

## 6. Voice model and evaluation

**The task:** given one clip's 47 features, predict which of the 4 members is speaking.

**Two candidate models:**
- **Logistic regression** (with `StandardScaler`): a simple, fast linear model whose probabilities are usually well calibrated. That matters because section 7 sets a threshold on those probabilities. The scaler is needed because features are on very different scales (roll-off in thousands of Hz, ZCR between 0 and 1).
- **Random forest**: many decision trees voting. It handles non-linear patterns and needs no scaling, but with this little data its probabilities tend to be less sharp.

**How we test it fairly: leave one recording out.** With only 16 recordings, a single train/test split would be too noisy. Instead, each round hides **one original recording and all 5 of its augmented copies**, trains on everything else, and predicts the hidden recording. Repeat for all 16 recordings, so every recording gets tested exactly once by a model that never saw it or its copies. This is `LeaveOneGroupOut` with `source_audio` as the group.

**Metrics:**
- **Accuracy:** the share of clips assigned to the right person.
- **Macro F1:** balances precision and recall per member, then averages across members equally, so one member can't hide another's poor results.
- **Log loss:** punishes confident wrong answers. Lower is better, and guessing 25% for every member scores about 1.39.

We report results on **original recordings only** as the honest number, because that's what real use looks like. Augmented rows are shown too, for comparison. The model with the best macro F1 on originals is chosen.

In [ ]:
cv = {}
for name, model in ap.candidate_models().items():
    metrics, results = ap.cross_validate(features, model)
    cv[name] = (metrics, results)

summary = pd.DataFrame({(name, subset): m for name, (metrics, _) in cv.items() for subset, m in metrics.items()}).T.round(3)
display(summary)

best_name = max(cv, key=lambda n: cv[n][0]["originals_only"]["f1_macro"])
best_results = cv[best_name][1]
print("Chosen model:", best_name)

In [ ]:
originals = best_results[best_results["augmentation"] == "original"]
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay.from_predictions(originals["member"], originals["predicted"], ax=ax, colorbar=False, cmap="Blues")
ax.set(title=f"Voice model ({best_name}), held-out original recordings", xlabel="Predicted speaker", ylabel="Actual speaker")
plt.tight_layout(); plt.savefig(FIG_DIR / "audio_confusion_matrix.png", dpi=150); plt.show()

**Interpretation:** _(fill in after running)_ Which model won, and by how much? Which members get confused with each other, and does that match what you saw in the MFCC profiles? Are scores on augmented rows higher or lower than on originals, and why? Keep in mind that with 4 recordings per person each misclassification moves accuracy by about 6%, so treat these numbers as rough estimates.

## 7. Confidence threshold: keeping outsiders out

The model always picks one of the 4 members, even for a stranger, because its probabilities have to add up to 100%. Picture a bouncer whose guest list has only four names, and who has to match every person at the door to one of them. The **threshold** lets the bouncer say "you don't look enough like anyone on the list."

**How it's chosen:**
- **Genuine confidences:** for each member recording the model got right in section 6, the confidence it had. These are what real members score.
- **Outsider confidences:** the final model's top confidence for each outsider clip.
- We try thresholds from 0.30 to 0.95 and pick the one with the best **balanced accuracy**: the average of "share of members let in" and "share of outsiders kept out". Ties go to the lower value, so members aren't rejected unnecessarily.

**No marking our own exam:** outsiders are split by *person*. Half are used to choose the threshold, and the other half stay hidden until the final check below. If those hidden outsiders are kept out too, the threshold generalises.

One limitation to state honestly in the report: a model trained on 4 people can still be confidently wrong about a stranger who sounds like one of them. The threshold reduces that risk; it doesn't remove it. That's why the system also needs the face check before the voice check.

In [ ]:
final_model = ap.train_final_model(features, ap.candidate_models()[best_name])
cols = ap.feature_columns(features)
genuine_conf = best_results.query("augmentation == 'original' and correct")["confidence"].to_numpy()

outsiders = ap.build_feature_table(ap.UNAUTH_AUDIO_DIR, outsiders=True, do_augment=False) \
    if any(ap.UNAUTH_AUDIO_DIR.glob("*.wav")) else pd.DataFrame()
if len(outsiders):
    tune_df, test_df = ap.split_outsiders(outsiders)
    tune_conf = final_model.predict_proba(tune_df[cols]).max(axis=1)
    test_conf = final_model.predict_proba(test_df[cols]).max(axis=1) if len(test_df) else np.array([])
    print("Tuning outsiders:", sorted(tune_df["member"].unique()), "| held-out outsiders:", sorted(test_df["member"].unique()))
else:
    tune_conf, test_conf = None, np.array([])
    print("No outsider clips found in data/unauthorized/audio, so falling back to the genuine-only rule.")

threshold, how = ap.choose_threshold(genuine_conf, tune_conf)
print(f"Threshold = {threshold:.2f}  ({how})")
print(f"Members accepted at this threshold: {(genuine_conf >= threshold).mean():.0%}")
if len(test_conf):
    print(f"Held-out outsiders rejected:       {(test_conf < threshold).sum()}/{len(test_conf)}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
bins = np.linspace(0.25, 1, 16)
ax.hist(genuine_conf, bins=bins, alpha=0.7, label="members (correct, held out)")
if tune_conf is not None:
    ax.hist(tune_conf, bins=bins, alpha=0.7, label="outsiders: tuning")
if len(test_conf):
    ax.hist(test_conf, bins=bins, alpha=0.7, label="outsiders: held out")
ax.axvline(threshold, color="black", linestyle="--", label=f"threshold = {threshold:.2f}")
ax.set(title="Model confidence: members vs outsiders", xlabel="Top predicted probability", ylabel="Number of clips")
ax.legend(); plt.tight_layout(); plt.savefig(FIG_DIR / "audio_threshold.png", dpi=150); plt.show()

**Interpretation:** _(fill in after running)_ How well do the two groups separate? Where does the threshold sit, and what does it cost (members rejected) versus what it protects against (outsiders accepted)? For a payment approval, which mistake is worse: a member having to try again, or a stranger getting in? Use that to justify the threshold.

## 8. Save the model and test it the way the app will

The saved bundle holds the trained model, the exact feature column order and the threshold, so the app can't accidentally use a different setup.

`ap.verify_speaker(path, expected_member)` is what the command-line app calls after the face model has recognised someone. It accepts the voice only if the predicted speaker **is that same member** and the confidence is at least the threshold. Without the same-member check, one member's face plus another member's voice would get through.

The three checks below are the three cases the demo needs to show.

In [ ]:
ap.save_bundle(final_model, cols, threshold)
print("Saved ->", ap.MODEL_PATH.relative_to(ROOT))
bundle = __import__("joblib").load(ap.MODEL_PATH)

a, b = members[0], members[1]
genuine_clip = sorted((ap.AUDIO_DIR / a).glob("*.wav"))[0]
checks = [("Genuine: right member, own voice", genuine_clip, a),
          ("Mismatch: face says another member", genuine_clip, b)]
outsider_clips = sorted(ap.UNAUTH_AUDIO_DIR.glob("*.wav"))
if outsider_clips:
    checks.append(("Outsider claiming to be a member", outsider_clips[-1], a))

for label, clip, claimed in checks:
    ok, predicted, conf = ap.verify_speaker(clip, claimed, bundle)
    print(f"{label:38} clip={clip.name:34} claimed={claimed:8} -> predicted={predicted:8} conf={conf:.2f}  {'APPROVED' if ok else 'DENIED'}")

**Note:** the genuine clip above was also used in training, so passing it only shows the plumbing works. Its real accuracy is the held-out result from section 6. For the demo video, record **one new clip** of yourself that the model has never heard, and show that it gets approved.